# Voice & Media Studio — Google Colab GPU Runner

**Capabilities:** Whisper ASR Transcription (`transcribe`) -> Script Prep (`prepare`) -> Edge-TTS (`tts`) -> Gawr Gura / Custom RVC v2 (`convert`) -> Audio QA (`qa`) -> Reveal Alignment (`align`) -> 1080p Render (`render`).

**Scope & Attribution:** Configured for internal / non-commercial educational production. Retain `THIRD_PARTY_LICENSES.md` and `licenses/` with distributed deliverables.

**Supported Colab Modes:**
1. **Direct Interactive Full-Compute**: Upload `exports/voice_studio_colab_bundle.zip` (or use cloned workspace / Drive), set your job config or input script/audio, and execute via `studio.py`.
2. **Drive Worker Mode (`drive_worker`)**: Connects to `python studio.py run --target colab_drive` from your PC, executes jobs on Colab GPU, and syncs deliverables + SHA-256 checksums back.
3. **Authenticated HTTP Agent (`http_agent`)**: Starts `python studio.py colab agent --host 0.0.0.0` behind your encrypted tunnel with Bearer token authentication.

In [ ]:
#@title 1. Setup Workspace, Inspect Colab GPU/CPU & Install Dependencies
import os, psutil, shutil, subprocess, zipfile
import torch

BUNDLE_ZIP = "voice_studio_colab_bundle.zip"  #@param {type:"string"}
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_BUNDLE_PATH = "/content/drive/MyDrive/VoiceStudioColab/voice_studio_colab_bundle.zip"  #@param {type:"string"}

if os.path.isfile("studio.py") and os.path.isdir("src/voice_studio"):
    WORK_ROOT = os.getcwd()
elif os.path.isfile("/content/webappdoc/studio.py") and not os.path.isfile("/content/webappdoc_prod/studio.py"):
    WORK_ROOT = "/content/webappdoc"
else:
    WORK_ROOT = "/content/webappdoc_prod"
os.makedirs(WORK_ROOT, exist_ok=True)
os.chdir(WORK_ROOT)

cpu_cores = os.cpu_count() or 2
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"CPU cores: {cpu_cores} | System RAM: {ram_gb:.1f} GB")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.1f} GB | CC: {cc[0]}.{cc[1]}")
else:
    print("WARNING: No CUDA GPU detected! Select Runtime -> Change runtime type -> T4 / L4 / A100 GPU.")

os.environ["OMP_NUM_THREADS"] = str(cpu_cores)
os.environ["MKL_NUM_THREADS"] = str(cpu_cores)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for dp in [DRIVE_BUNDLE_PATH, "/content/drive/MyDrive/VoiceStudioColab/voice_studio_colab_bundle.zip", "/content/drive/MyDrive/voice_studio_colab_bundle.zip"]:
        if dp and os.path.exists(dp):
            shutil.copy2(dp, os.path.join(WORK_ROOT, os.path.basename(BUNDLE_ZIP)))
            break
elif os.path.isfile(os.path.join(WORK_ROOT, "studio.py")):
    print(f"Detected existing studio workspace in {WORK_ROOT} (studio.py present); skipping bundle upload.")
elif not os.path.exists(BUNDLE_ZIP):
    for cand in [BUNDLE_ZIP, os.path.join("/content", os.path.basename(BUNDLE_ZIP)), "/content/voice_studio_colab_bundle.zip", "/content/drive/MyDrive/VoiceStudioColab/voice_studio_colab_bundle.zip"]:
        if os.path.exists(cand):
            BUNDLE_ZIP = cand
            break
    else:
        from google.colab import files
        print("Upload exports/voice_studio_colab_bundle.zip...")
        uploaded = files.upload()
        if uploaded:
            BUNDLE_ZIP = list(uploaded.keys())[0]

if os.path.exists(BUNDLE_ZIP):
    with zipfile.ZipFile(BUNDLE_ZIP, "r") as zf:
        zf.extractall(WORK_ROOT)

!apt-get -qq update && apt-get -qq install -y ffmpeg aria2 > /dev/null
!pip -q install -U edge-tts faster-whisper transformers huggingface_hub onnxruntime-gpu faiss-cpu soxr einops pyloudnorm librosa soundfile pillow scipy PyYAML python-pptx pypdf
print("Dependencies installed and studio ready:", WORK_ROOT)

In [ ]:
#@title 2. Pre-Cache Gawr Gura / Custom RVC + Applio & Verify Video Encoder
import os, sys

WORK_ROOT = globals().get("WORK_ROOT") or ("/content/webappdoc" if os.path.isfile("/content/webappdoc/studio.py") else "/content/webappdoc_prod")
os.chdir(WORK_ROOT)
sys.path.insert(0, os.path.join(WORK_ROOT, "src"))

assert os.path.isfile("THIRD_PARTY_LICENSES.md") and os.path.isfile("licenses/GawrGura-RVC_README.md")
print("Verified license notices in THIRD_PARTY_LICENSES.md and licenses/")

VOICE_PROFILE = "gawr_gura"  #@param ["gawr_gura", "direct_tts"]
if VOICE_PROFILE == "gawr_gura":
    from voice_studio.audio.rvc_assets import default_rvc_dirs, ensure_applio_repo, ensure_rvc_model_assets
    from voice_studio.config.voices import build_voice_profile, profile_to_rvc_dict
    rvc_cfg = profile_to_rvc_dict(build_voice_profile({"profile": "gawr_gura"}))
    model_dir, applio_dir = default_rvc_dirs("gawr_gura")
    pth_path, idx_path = ensure_rvc_model_assets("gawr_gura", model_dir=model_dir, rvc_cfg=rvc_cfg)
    ensure_applio_repo(applio_dir)
    print(f"Ready [gawr_gura]: pth={pth_path}, idx={idx_path}")

from voice_studio.video.compositor import detect_best_video_encoder
print(f"Auto-detected Video Encoder: {detect_best_video_encoder()}")

In [ ]:
#@title 3. Optional Remote Bridge Mode: Drive Worker OR HTTP Agent
import os, secrets, subprocess

BRIDGE_MODE = "disabled"  #@param ["disabled", "drive_worker", "http_agent"]
DRIVE_ROOT = "/content/drive/MyDrive"  #@param {type:"string"}
AGENT_PORT = 8765  #@param {type:"integer"}
AGENT_TOKEN = ""  #@param {type:"string"}

WORK_ROOT = globals().get("WORK_ROOT") or os.getcwd()
os.chdir(WORK_ROOT)
if BRIDGE_MODE == "drive_worker":
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run(["python3", "studio.py", "colab", "drive-worker", "--drive-mount-dir", DRIVE_ROOT, "--watch"], check=True)
elif BRIDGE_MODE == "http_agent":
    token = AGENT_TOKEN or secrets.token_urlsafe(24)
    print(f"Starting Voice Studio Colab Agent on port {AGENT_PORT} | Bearer Token: {token}")
    print("Run locally over HTTPS tunnel: python studio.py run --target colab_agent --colab-url <HTTPS_URL> --colab-token <TOKEN>")
    subprocess.run(["python3", "studio.py", "colab", "agent", "--host", "0.0.0.0", "--port", str(AGENT_PORT), "--colab-token", token], check=True)
else:
    print("Bridge mode disabled; proceed to Cells 4-6 for direct interactive Colab execution.")

In [ ]:
#@title 4. Execute Full Workflow via studio.py (Custom Script, Media ASR, or Included Job Config)
import os, subprocess

WORK_ROOT = globals().get("WORK_ROOT") or os.getcwd()
os.chdir(WORK_ROOT)

CONFIG_PATH = "configs/colab_pro.yaml"  #@param ["configs/colab_pro.yaml", "configs/studio.yaml", "configs/custom_job.yaml"]
INPUT_OVERRIDE = ""  #@param {type:"string"}
JOB_ID = "colab_pro_comparison"  #@param {type:"string"}
VOICE_PROFILE = globals().get("VOICE_PROFILE", "gawr_gura")
AUDIO_ONLY = False  #@param {type:"boolean"}
RUN_ASR_CHECK = True  #@param {type:"boolean"}
FORCE_FRESH = False  #@param {type:"boolean"}

cmd = [
    "python3", "studio.py", "run",
    "--config", CONFIG_PATH,
    "--voice", VOICE_PROFILE,
    "--job", JOB_ID,
    "--target", "local",
]
if INPUT_OVERRIDE.strip():
    cmd.extend(["--input", INPUT_OVERRIDE.strip()])
if AUDIO_ONLY:
    cmd.append("--audio-only")
if FORCE_FRESH:
    cmd.append("--no-resume")
print("Executing:", " ".join(cmd))
import json, time, psutil, torch
t0 = time.monotonic()
subprocess.run(cmd, check=True)
elapsed_s = round(time.monotonic() - t0, 2)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if torch.cuda.is_available() else 0.0
if RUN_ASR_CHECK:
    wav_cand = os.path.join(WORK_ROOT, f"output/{JOB_ID}/deliverables")
    import glob
    wavs = sorted(glob.glob(os.path.join(wav_cand, "*.wav")))
    if wavs:
        subprocess.run(["python3", "studio.py", "transcribe", "--config", CONFIG_PATH, "--input", wavs[0], "--out", f"output/{JOB_ID}/transcript"], check=True)
meta_out = {"config_path": CONFIG_PATH, "job_id": JOB_ID, "gpu": gpu_name, "vram_gb": vram_gb, "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2), "cpu_cores": os.cpu_count(), "elapsed_sec": elapsed_s}
with open(os.path.join(WORK_ROOT, f"output/{JOB_ID}/colab_runtime_report.json"), "w", encoding="utf-8") as f:
    json.dump(meta_out, f, indent=2)
print("Colab Runtime Report:", json.dumps(meta_out))

In [ ]:
#@title 5. Inspect Transcript (If Media Input) & Audition Sample Audio Inline
import os
from IPython.display import Audio, display

WORK_ROOT = globals().get("WORK_ROOT") or os.getcwd()
JOB_ID = globals().get("JOB_ID", "colab_studio_run")
tr_txt = os.path.join(WORK_ROOT, f"output/{JOB_ID}/transcript/transcript.txt")
if os.path.exists(tr_txt):
    print("=== TRANSCRIPT ===")
    print(open(tr_txt, encoding="utf-8").read()[:1500])

sample_wav = os.path.join(WORK_ROOT, f"output/{JOB_ID}/audio_slides_wav/slide_01.wav")
if os.path.exists(sample_wav):
    print(f"Previewing {sample_wav}:")
    display(Audio(sample_wav))

In [ ]:
#@title 6. Package & Download Completed Job Deliverables (WAV, MP3, MP4, SRT/TXT, HTML Players)
import glob, hashlib, os, zipfile
from google.colab import files

WORK_ROOT = globals().get("WORK_ROOT") or os.getcwd()
os.chdir(WORK_ROOT)
JOB_ID = globals().get("JOB_ID", "colab_studio_run")
out_archive = f"{JOB_ID}_Colab_Deliverables.zip"
with zipfile.ZipFile(out_archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in ["THIRD_PARTY_LICENSES.md"]:
        if os.path.exists(fn):
            zf.write(fn)
    for pattern in ["licenses/*", f"output/{JOB_ID}/deliverables/**/*", f"output/{JOB_ID}/audio_slides_wav/**/*", f"output/{JOB_ID}/transcript/**/*", f"output/{JOB_ID}/*.json"]:
        for fp in sorted(glob.glob(pattern, recursive=True)):
            if os.path.isfile(fp):
                zf.write(fp)
sha = hashlib.sha256(open(out_archive, "rb").read()).hexdigest()
if os.path.isdir("/content/drive/MyDrive"):
    import shutil
    drive_out = "/content/drive/MyDrive/VoiceStudioColab/outputs"
    os.makedirs(drive_out, exist_ok=True)
    shutil.copy2(out_archive, os.path.join(drive_out, out_archive))
    print(f"Synced deliverables archive to Google Drive: {drive_out}/{out_archive}")
print(f"Ready: {out_archive} ({os.path.getsize(out_archive)/(1024**2):.2f} MB) | SHA-256: {sha}")
files.download(out_archive)